# Data Cleaning

Notebook ini menjalankan fungsi pembersihan dari `src/data_cleaning.py`. File di `data/raw` hanya dibaca. Hasil dan log perubahan ditulis ke `data/processed` dalam format CSV.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in project_candidates if (p / 'data' / 'raw').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Root project dengan data/raw tidak ditemukan.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_cleaning import (
    clean_bi_rate, clean_index_lq45, clean_inflation, clean_pdb,
    clean_pmi, clean_prices, clean_summary_metrics, clean_trade_balance,
    clean_usdidr, write_csv,
)

def read_raw(name):
    path = RAW_DIR / name
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    return pd.read_csv(path)

adj_close = read_raw('adj_close_10y.csv')
index_lq45_raw = read_raw('index_lq45_10y.csv')
usdidr_raw = read_raw('usdidr_10y.csv')
pdb_raw = read_raw('Data PDB 2016-2026.csv')
inflation_raw = read_raw('Tingkat Inflasi Indonesia ( M - to - M).csv')
trade_raw = read_raw('Neraca Perdagangan 2016 - 2026.csv')
pmi_raw = read_raw('PMI Manufacturing.csv')
bi_rate_raw = read_raw('BI-7Day-RR_Sorted.xlsx')
summary_raw = read_raw('summary_metrics.csv')
trading_calendar = pd.DatetimeIndex(pd.to_datetime(index_lq45_raw['Date'])).sort_values()
correction_path = RAW_DIR / 'koreksi_manual.csv'
if not correction_path.exists():
    correction_path = RAW_DIR / 'correction_manual.csv'
cleaning_log = []
display(Markdown(f'**Raw files loaded:** 9 &nbsp;&nbsp; **Official trading dates:** {len(trading_calendar)}'))

**Raw files loaded:** 9 &nbsp;&nbsp; **Official trading dates:** 2411

In [2]:
prices = clean_prices(adj_close, index_lq45_raw, cleaning_log)
index_lq45 = clean_index_lq45(index_lq45_raw, trading_calendar, cleaning_log)
usdidr = clean_usdidr(usdidr_raw, trading_calendar, cleaning_log)
pdb = clean_pdb(pdb_raw, correction_path if correction_path.exists() else None, cleaning_log)
inflation = clean_inflation(inflation_raw, cleaning_log)
trade_balance = clean_trade_balance(trade_raw, cleaning_log)
pmi_bi = clean_pmi(pmi_raw, cleaning_log)
bi_rate = clean_bi_rate(bi_rate_raw, cleaning_log)
summary_metrics = clean_summary_metrics(summary_raw, cleaning_log)
if not correction_path.exists():
    cleaning_log.append({
        'dataset': 'pdb',
        'action': 'manual_correction',
        'detail': 'correction_manual.csv tidak ditemukan; tidak ada koreksi manual diterapkan.',
        'affected_rows': 0,
    })

outputs = {
    'prices.csv': prices,
    'index_lq45.csv': index_lq45,
    'usdidr.csv': usdidr,
    'pdb.csv': pdb,
    'inflation.csv': inflation,
    'trade_balance.csv': trade_balance,
    'pmi_bi.csv': pmi_bi,
    'bi_rate.csv': bi_rate,
    'summary_metrics.csv': summary_metrics,
}
for filename, frame in outputs.items():
    write_csv(frame, PROCESSED_DIR / filename)

cleaning_log_df = pd.DataFrame(cleaning_log, columns=['dataset', 'action', 'detail', 'affected_rows'])
write_csv(cleaning_log_df, PROCESSED_DIR / 'cleaning_log.csv')
display(cleaning_log_df)

,dataset,action,detail,affected_rows
0,prices,forward_fill,2019-06-19; limit=1,5
1,prices,forward_fill,2026-08-25; limit=1,3
2,prices,drop_exchange_holidays,Dikeluarkan dari kalender resmi: 54 tanggal.,54
3,prices,drop_zero_volume_tail,Baris terakhir 2026-09-22 dibuang karena Volum...,1
4,usdidr,forward_fill,Reindex kalender resmi; limit=3 hari.,24
5,pdb,unit_conversion,Nilai ADHK koreksi manual dikonversi dari mili...,42
6,pdb,manual_correction,Menggunakan koreksi_manual.csv.,126
7,trade_balance,unit_standardization,Kolom disimpan sebagai neraca_ribu_usd setelah...,120
8,summary_metrics,drop_aggregate_row,Menghapus Ticker ^JKLQ45.,1
9,summary_metrics,null_bad_scale_itmg,"ITMG.JK PBV, EV / EBITDA, dan EV / Revenue dit...",3


## Validasi output

In [3]:
summary = pd.DataFrame([
    {'file': name, 'rows': len(frame), 'columns': len(frame.columns), 'date_min': frame['date'].min() if 'date' in frame else None, 'date_max': frame['date'].max() if 'date' in frame else None}
    for name, frame in outputs.items()
])
display(summary)

# prices memakai kalender resmi, tanpa tanggal libur dan tanpa tail zero-volume.
assert len(prices) == len(trading_calendar) - 1
assert pd.Timestamp('2026-08-25') not in set(prices['date'])
assert prices['date'].is_monotonic_increasing and not prices['date'].duplicated().any()
assert prices.loc[prices['date'].eq('2019-06-19')].notna().all().all()

# Kurs memiliki kalender yang sama dan tidak menyisakan gap setelah ffill limit=3.
assert usdidr['date'].equals(index_lq45['date'])
assert usdidr['date'].is_monotonic_increasing and not usdidr['date'].duplicated().any()

# PDB/PMI memiliki tanggal akhir kuartal.
assert pdb['date'].dt.is_quarter_end.all()
assert pmi_bi['date'].dt.is_quarter_end.all()

# Validasi khusus summary metrics.
assert '^JKLQ45' not in set(summary_metrics['Ticker'])
itmg = summary_metrics.loc[summary_metrics['Ticker'].eq('ITMG.JK')].iloc[0]
assert itmg[['Price to Book (PBV)', 'EV / EBITDA', 'EV / Revenue']].isna().all()

display(Markdown(f'**Selesai:** {len(outputs) + 1} file ditulis ke `{PROCESSED_DIR}`.'))
display(cleaning_log_df)

,file,rows,columns,date_min,date_max
0,prices.csv,2410,16,2016-09-22,2026-09-21
1,index_lq45.csv,2411,7,2016-09-22,2026-09-22
2,usdidr.csv,2411,7,2016-09-22,2026-09-22
3,pdb.csv,42,4,2016-03-31,2026-06-30
4,inflation.csv,121,2,2016-08-31,2026-08-31
5,trade_balance.csv,120,2,2016-08-31,2026-07-31
6,pmi_bi.csv,42,2,2016-03-31,2026-06-30
7,bi_rate.csv,122,2,2016-09-22,2026-08-19
8,summary_metrics.csv,15,20,NaT,NaT


**Selesai:** 10 file ditulis ke `D:\7. PROYEK\5. BACKTEST & OPTIMIZE\Ori_Data\data\processed`.

,dataset,action,detail,affected_rows
0,prices,forward_fill,2019-06-19; limit=1,5
1,prices,forward_fill,2026-08-25; limit=1,3
2,prices,drop_exchange_holidays,Dikeluarkan dari kalender resmi: 54 tanggal.,54
3,prices,drop_zero_volume_tail,Baris terakhir 2026-09-22 dibuang karena Volum...,1
4,usdidr,forward_fill,Reindex kalender resmi; limit=3 hari.,24
5,pdb,unit_conversion,Nilai ADHK koreksi manual dikonversi dari mili...,42
6,pdb,manual_correction,Menggunakan koreksi_manual.csv.,126
7,trade_balance,unit_standardization,Kolom disimpan sebagai neraca_ribu_usd setelah...,120
8,summary_metrics,drop_aggregate_row,Menghapus Ticker ^JKLQ45.,1
9,summary_metrics,null_bad_scale_itmg,"ITMG.JK PBV, EV / EBITDA, dan EV / Revenue dit...",3
